**SCRAPPING KOMENTAR YOUTUBE**

**Ekstraksi Opini Publik: Sentimen Netizen Terhadap Pencopotan Menkeu Purbaya**

Wacana perombakan Kabinet Merah Putih mencapai titik didihnya melalui manuver pencopotan Purbaya dari kursi Menteri Keuangan. Dinamika ini bukan sekadar pergantian kursi birokrasi, melainkan transisi krusial yang menggetarkan fondasi kebijakan fiskal negara. Ruang publik digital kini berubah menjadi episentrum perdebatan tajam yang secara langsung merepresentasikan ketidakpastian pasar, polarisasi politik elit, dan fluktuasi tingkat kepercayaan masyarakat.


Proyek web scraping ini dirancang secara sistematis untuk merekam, menstrukturkan, dan mengamankan respons organik masyarakat dari kolom komentar YouTube.

**🛠️ Spesifikasi Teknis Ekstraksi Data**


Metodologi: Penarikan data menggunakan endpoint resmi YouTube Data API v3 guna menjamin validitas dan struktur data.

Variabel Data Terpusat: Mengekstrak empat feature utama, yaitu Username, Komentar (teks murni yang telah dibersihkan dari line-break), Waktu (timestamp publikasi), dan Likes (amplifikasi dukungan).

Pembatasan Volume: Pengambilan data dibatasi secara mutlak pada angka maksimal 600 komentar teratas untuk setiap tautan video guna menjaga keseimbangan dan relevansi dataset.

Format Keluaran (Output): Data dikonsolidasikan ke dalam satu file tunggal berformat Comma Separated Values (.csv) yang siap untuk memasuki tahap data preprocessing.

**🔗 Daftar Sumber Tautan Video (Dataset Target)**


Korpus teks akan diekstrak secara otomatis dari 5 (lima) representasi liputan digital berikut:

Target Data 1: https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4

Target Data 2: https://youtu.be/ORpLeh26-xU?si=1WgN-c4Mfp7OYqvC

Target Data 3: https://youtu.be/2DMsAllPk8s?si=7uhqPDCcqZSApexs

Target Data 4: https://youtu.be/lrJ-00vrFd4?si=o4_TGN35vOHXus26

Target Data 5: https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q

In [ ]:
import requests
import csv
import re

# 1. MASUKKAN API KEY ANDA DI SINI
API_KEY = "API_KEY"

# 2. MASUKKAN 6 LINK VIDEO DI SINI
DAFTAR_LINK = [
    "https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4",
    "https://youtu.be/ORpLeh26-xU?si=1WgN-c4Mfp7OYqvC",
    "https://youtu.be/2DMsAllPk8s?si=7uhqPDCcqZSApexs",
    "https://youtu.be/lrJ-00vrFd4?si=o4_TGN35vOHXus26",
    "https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q"
]

MAKSIMAL_KOMENTAR_PER_VIDEO = 600

def dapatkan_video_id(url):
    """Fungsi untuk mengekstrak Video ID dari link YouTube secara otomatis."""
    # Mencari pola ID video dari berbagai format link YouTube
    match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11}).*", url)
    if match:
        return match.group(1)
    return None

semua_data = []

print("=== MEMULAI PROSES SCRAPING KOMENTAR YOUTUBE ===")

# Looping / Perulangan untuk setiap link video
for url in DAFTAR_LINK:
    video_id = dapatkan_video_id(url)

    if not video_id:
        print(f"[-] URL tidak valid, dilewati: {url}")
        continue

    print(f"\n[*] Memproses video: {video_id} ({url})")

    api_url = "https://www.googleapis.com/youtube/v3/commentThreads"
    params = {
        "part": "snippet",
        "videoId": video_id,
        "maxResults": 100,  # Sekali tarik ambil 100 komentar (batas maksimal API)
        "textFormat": "plainText",
        "key": API_KEY,
    }

    jumlah_diambil = 0

    # Looping untuk mengambil halaman (pagination) sampai target terpenuhi
    while True:
        try:
            response = requests.get(api_url, params=params)
            data = response.json()

            # Jika terjadi error dari API (misal: komentar dinonaktifkan)
            if "error" in data:
                print(f"    [!] Error pada video {video_id}: {data['error']['message']}")
                break

            items = data.get("items", [])

            for item in items:
                # Cek jika sudah mencapai batas 500
                if jumlah_diambil >= MAKSIMAL_KOMENTAR_PER_VIDEO:
                    break

                snippet = item["snippet"]["topLevelComment"]["snippet"]

                # Memasukkan data ke dalam list
                semua_data.append({
                    "Link Video": url,
                    "Username": snippet.get("authorDisplayName", "Tidak Diketahui"),
                    "Komentar": snippet.get("textDisplay", "").replace("\r", " ").replace("\n", " "), # Membersihkan enter/baris baru
                    "Waktu": snippet.get("publishedAt", ""),
                    "Likes": snippet.get("likeCount", 0)
                })
                jumlah_diambil += 1

            # Jika sudah 500 komentar, hentikan pencarian untuk video ini
            if jumlah_diambil >= MAKSIMAL_KOMENTAR_PER_VIDEO:
                print(f"    [+] Selesai: Mencapai target {MAKSIMAL_KOMENTAR_PER_VIDEO} komentar.")
                break

            # Mengecek apakah masih ada halaman komentar selanjutnya (next page)
            if "nextPageToken" in data:
                params["pageToken"] = data["nextPageToken"]
            else:
                print(f"    [+] Selesai: Semua komentar telah diambil (Total: {jumlah_diambil} komentar).")
                break

        except Exception as e:
            print(f"    [!] Terjadi kesalahan sistem: {e}")
            break

# 3. MENYIMPAN HASIL KE DALAM FILE CSV
nama_file = "hasil_scraping_5_video.csv"
if semua_data:
    print("\n=== MENYIMPAN DATA ===")
    with open(nama_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["Link Video", "Username", "Komentar", "Waktu", "Likes"])
        writer.writeheader()
        writer.writerows(semua_data)
    print(f"[V] BERHASIL! Total keseluruhan {len(semua_data)} komentar telah disimpan ke dalam file '{nama_file}'.")
else:
    print("\n[!] Tidak ada data komentar yang berhasil diambil.")

=== MEMULAI PROSES SCRAPING KOMENTAR YOUTUBE ===

[*] Memproses video: MdREnoDpYew (https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4)
    [+] Selesai: Mencapai target 600 komentar.

[*] Memproses video: ORpLeh26-xU (https://youtu.be/ORpLeh26-xU?si=1WgN-c4Mfp7OYqvC)
    [+] Selesai: Mencapai target 600 komentar.

[*] Memproses video: 2DMsAllPk8s (https://youtu.be/2DMsAllPk8s?si=7uhqPDCcqZSApexs)
    [+] Selesai: Semua komentar telah diambil (Total: 175 komentar).

[*] Memproses video: lrJ-00vrFd4 (https://youtu.be/lrJ-00vrFd4?si=o4_TGN35vOHXus26)
    [+] Selesai: Mencapai target 600 komentar.

[*] Memproses video: Qr9vSphA6y4 (https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q)
    [+] Selesai: Mencapai target 600 komentar.

=== MENYIMPAN DATA ===
[V] BERHASIL! Total keseluruhan 2575 komentar telah disimpan ke dalam file 'hasil_scraping_5_video.csv'.


In [ ]:
# akan ditampilkan dalam bentuk dataframe pandas sebagai bukti bahwa scrapping berhasil
import pandas as pd
df = pd.read_csv('hasil_scraping_5_video.csv')
df

,Link Video,Username,Komentar,Waktu,Likes
0,https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4,@paltisambungan,Pemecatan Purbaya sesuai dengan perintah yg me...,2026-09-28T02:55:44Z,0
1,https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4,@NoviRahmawati-kh5pe,"Pa purbaya,pokoknya ga akan lupa dech pa...sel...",2026-09-25T16:16:01Z,0
2,https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4,@NinaAlnina,😢😢😢,2026-09-24T04:32:11Z,0
3,https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4,@igesaliibrahim,Orang baik dan jujur di benci orang2 pemerintahan,2026-09-24T03:14:36Z,0
4,https://youtu.be/MdREnoDpYew?si=En-OV-CAL5MTydt4,@mizzymeisje5519,kdm juga ga perlu jadi presiden.presiden cuma ...,2026-09-23T17:47:07Z,0
...,...,...,...,...,...
2570,https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q,@Comet-Dy,"Kencangkan sabuk pengaman kawan, ombak pajak a...",2026-09-15T00:42:23Z,0
2571,https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q,@golden-19B,"Bea cukai, mafia, koruptor,, tepuk tangan👏, me...",2026-09-15T00:40:19Z,0
2572,https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q,@muhammadqomaruddin6616,Mantan Menkeu Sri Mulyani aman di dua presiden...,2026-09-15T00:40:17Z,0
2573,https://youtu.be/Qr9vSphA6y4?si=cG9IWvq8tMRGtj9q,@umanthea9995,Lebih baik keluar drpd kerja dilingkunga para ...,2026-09-15T00:39:27Z,0


In [ ]:
# catatan
# kode tersebut jila dijalankan ulang 'ada' kemungkinan akan eror,
# hal ini dikarenakan api key akan saya revoke untuk menghindari tagihan biaya yang tidak terduga

In [ ]:
# NAMA : MUHAMMAD IRFAN BAIHAQI
# NPM : 5230411327

In [ ]:
# ----------------------&&&-----------------------------